# Franken training

In this notebook we will explore the training of `franken` on a small dataset of DFT calculations.

We use the H2O data obtained from DFT calculations (using RPBE+D3 theory), originally collected by [Montero de Hijes et al.](https://doi.org/10.1063/5.0197105).

Check the [documentation](https://franken.readthedocs.io/) for a description of each argument.

This notebook is also available on [Google Colab](https://colab.research.google.com/github/CSML-IIT-UCL/franken/blob/main/notebooks/training.ipynb) for easy running.

In [1]:
try:
    import franken
except ImportError:
    %pip install "franken[mace]"
    import franken

In [2]:
import json

import ase.io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from franken.autotune import autotune
from franken.backbones.utils import CacheDir
from franken.config import (
    AutotuneConfig,
    DatasetConfig,
    GaussianRFConfig,
    HPSearchConfig,
    MaceBackboneConfig,
    SolverConfig,
)
from franken.datasets.registry import DATASET_REGISTRY

## Load the data

The first step is defining the training and validation datasets. 
These should be extended XYZ (Franken expects ASE units: eV for energies, eV/Å for forces, and eV/Å³ for stress.) 

In this tutorial we use the water dataset used in the reference paper for which we have stored the paths in a internal dataset registry. The dataset will be downloaded into Franken's cache directory (`CacheDir.get()`).

In [3]:
# Define train and valid paths

train_path = DATASET_REGISTRY.get_path("water", "train", base_path=CacheDir.get())
val_path = DATASET_REGISTRY.get_path("water", "val", base_path=CacheDir.get())

print(f"Train path: {train_path}")
train_atoms = ase.io.read(train_path, index=0) # Print the first frame of the datasets
print(f"Train[0]: {train_atoms}")

print(f"\nValidation path: {val_path}")
val_atoms = ase.io.read(val_path, index=0)
print(f"Validation[0]: {val_atoms}")

Train path: /home/lbonati@iit.local/.franken/water/ML_AB_dataset_1.xyz
Train[0]: Atoms(symbols='H128O64', pbc=True, cell=[[13.101991, -1.0387e-05, -1.4377e-05], [0.0, 13.101991, 1.1531e-05], [0.0, 0.0, 13.101992]], calculator=SinglePointCalculator(...))

Validation path: /home/lbonati@iit.local/.franken/water/ML_AB_dataset_2-val.xyz
Validation[0]: Atoms(symbols='H128O64', pbc=True, cell=[[13.290949802, 2.2628e-05, 5.59e-06], [0.0, 13.290950224, -8.5923e-05], [0.0, -0.0, 13.290956559]], calculator=SinglePointCalculator(...))


TODO: specify that we use create a dataset config. Specify that for computational efficiency here we use max_train_samples = 8 (randomly sampled). this should be removed to train on the whole dataset. 

In [4]:
dataset_cfg = DatasetConfig(
    train_path=str(train_path), # e.g. "train.xyz"
    val_path=str(val_path),
    max_train_samples=8,
)

## franken' ingredients

`franken` fits a potential by combining three components:
1. a **GNN backbone** (e.g. a pretrained MACE model)
2. a **kernel**, approximated with random features 
3. a **linear solver** which optimizes the weights of the model

### 1) Backbone

We need to configure the GNN backbone we wish to use since `franken` extracts pre-trained atomic features from it.
In the paper we used mainly the [**MACE MP0 small** backbone](https://github.com/ACEsuit/mace-foundations) (identified by its `mace_mh/0` ID). However, this can be used with many other backbones, see https://franken.readthedocs.io/topics/training_backbones.html. In particuular, here we use one of the most recents ones, mace_mh/0 

The first time they are used, the backboneb will be downloaded as necessary into the cache directory. 

> **Note:** The cache directory is used to store model backbones and downloaded datasets.
> It defaults to `$HOME/.franken` for the current user and can be configured by setting the
> `FRANKEN_CACHE_DIR` environment variable before importing Franken. For example:
> ```python
> import os
> os.environ["FRANKEN_CACHE_DIR"] = "/path/to/my/cache/franken-cache"
> ```

In [5]:
gnn_config = MaceBackboneConfig(path_or_id="mace_mh/0")

### 2) Kernel parameters (Random Features)

TODO: explain the two parameters which need to be set

no. RFs: no. parameters of the model, describes how accurate the RFs describe the kernel. a few thousands is typically ok. 
(4096 e' un overkill per 8 samples :) 

length-scale of the kernel. this is the resolution of the kernel. in principle it needs to be optimized, e.g. via grid search. One can do this  on a subset since it is expensive. Later, a manually specified value can be used.

In [6]:
rf_config = GaussianRFConfig(
    num_random_features=4096,
    length_scale=HPSearchConfig(values=[1.,5.,10.,20.,30.])
)

> **Tip — avoiding the length-scale search altogether:** `franken` also provides a `MultiscaleGaussianRFConfig` kernel, which combines several length scales into a *single* model (instead of training one model per length scale). This removes the need to search `length_scale` at all, at the cost of a somewhat larger random feature map:
> ```python
> from franken.config import MultiscaleGaussianRFConfig
>
> rf_config = MultiscaleGaussianRFConfig(
>     num_random_features=4096,
>     length_scale_low=4.0,
>     length_scale_high=24.0,
>     length_scale_num=4,
> )
> ```

### 3) Solver 

Force-weight: relative weight of the forces compared to energy (by default 1)

l2_regularization: magnitude of reg. acting on the weights (avoids overfitting)

Note that the two groups of parameters (Kernel and solver) are not equally expensive to search,
While optimizing the **Kernel parameters** is expensive,  for every value, the whole random feature map has to be recomputed and the model refit from scratch.for the **Solver parameters** this is much cheaper: once the feature map is computed, `autotune` can efficiently re-fit and re-score many combinations by reusing it.

Solver hyperparameters are inexpensive to search because the costly feature maps can be reused. We test six logarithmically spaced L2 penalties from $10^{-10}$ to $10^{-5}$ and seven force weights from $10^{-3}$ to $10^3$.

The energy weight defaults to 1.0, and target weights are normalized internally. The configured force weight can therefore be read as the force-to-energy weight ratio.

In [7]:
solver_cfg = SolverConfig(
    l2_penalty=HPSearchConfig(start=-11, stop=-6, num=6, scale="log"),
    force_weight=HPSearchConfig(start=-2, stop=4, num=10, scale="log"),
)

### Hyperparameter optimization (autotune)

`franken` provides an `autotune` utility that performs a grid-search over them and selects the best model according to a validation set. 

Settings:
- `run_dir` is the parent directory in which a unique run folder containing logs and checkpoints will be created.
- `jac_chunk_size`:  how any samples are processed simultaneosuly. if you get **RuntimeError: CUDA out of memory** try to lower the `jac_chunk_size` to 8, 16 ,32, 64, etc. instead of "auto".
- `metrics`: list of metrics to be evaluated  (see https://franken.readthedocs.io/topics/training_metrics.html)
- `best_model_selection`:   which metrics to use to choose the best model (default: both energy and force mae)


> **CLI version**: The configuration have built is equivalent to running from the command line:
> ```bash
> franken.autotune \
>     --train-path $HOME/.franken/water/ML_AB_dataset_1.xyz
>     --val-path $HOME/.franken/water/ML_AB_dataset_2-val.xyz
>     --max-train-samples 8 \
>     --jac-chunk-size "auto" \
>     --run-dir "./results" \
>     --backbone=mace --mace.path-or-id "mace_mh/0" \
>     --rf=gaussian --gaussian.num-rf 4096 --gaussian.length-scale="[1., 5., 10.0, 20.0, 30.0]"
> ```

> **Note — full manual control:** if you need to fully customize the training loop (e.g. custom batching, logging, or a training procedure not covered by `autotune`), `franken` also exposes the lower-level `FrankenPotential` model and `LowMemRandomFeaturesTrainer`/`RandomFeaturesTrainer` classes directly (see the [API reference](https://franken.readthedocs.io/) for details). 


In [8]:
autotune_cfg = AutotuneConfig(
    dataset=dataset_cfg,
    solver=solver_cfg,
    backbone=gnn_config,
    rfs=rf_config,
    metrics=["energy_MAE", "forces_MAE"],
    jac_chunk_size="auto", # Note: if you get **RuntimeError: CUDA out of memory** try to lower the `jac_chunk_size` to 8, 16 ,32, 64, etc. instead of "auto". 
    run_dir="./results",
)

run_path = autotune(autotune_cfg)

atomic_energies: None
console_logging_level: INFO
dtype: float64
eval_splits: None
jac_chunk_size: auto
rf_normalization: leading_eig
run_dir: ./results
save_every_model: False
save_fmaps: False
scale_by_species: True
seed: 1337
backbone:
    family: mace
    interaction_block: 2
    path_or_id: mace_mh/0
best_model_selection:
    []
dataset:
    max_train_samples: 8
    name: null
    test_path: null
    train_path: /home/lbonati@iit.local/.franken/water/ML_AB_dataset_1.xyz
    val_path: /home/lbonati@iit.local/.franken/water/ML_AB_dataset_2-val.xyz
metrics:
    - energy_MAE
    - forces_MAE
rfs:
    length_scale:
      num: null
      scale: null
      start: null
      stop: null
      value: null
      values:
      - 1.0
      - 5.0
      - 10.0
      - 20.0
      - 30.0
    num_random_features: 4096
    rf_type: gaussian
    rng_seed: 1337
    use_offset: true
solver:
    energy_weight: 1.0
    force_weight:
      num: 10
      scale: log
      start: -2
      stop: 4
      value

ASE -> Franken (val): 100%|██████████| 189/189 [00:00<00:00, 312.70it/s]


Computing dataset statistics:   0%|          | 0/8 [00:00<?, ?it/s]

2026-09-07 18:51:40.416 INFO (rank 0): jacobian chunk size automatically set to 16
2026-09-07 18:53:10.387 WARNING (rank 0): `leading_eig` normalization has high memory usage. If you encounter OOM errors try to disable it.
2026-09-07 18:59:17.111 INFO (rank 0): Trial   1 | rf_type: gaussian | num_random_features:  4096   | length_scale:  1.000  | use_offset:  True   | rng_seed:  1337   | Best trial 1 (energy 0.66 meV/atom) (forces 51.92 meV/Ang)
2026-09-07 18:59:18.864 INFO (rank 0): jacobian chunk size automatically set to 16
2026-09-07 19:00:48.752 WARNING (rank 0): `leading_eig` normalization has high memory usage. If you encounter OOM errors try to disable it.
2026-09-07 19:06:52.848 INFO (rank 0): Trial   2 | rf_type: gaussian | num_random_features:  4096   | length_scale:  5.000  | use_offset:  True   | rng_seed:  1337   | Best trial 2 (energy 0.16 meV/atom) (forces 12.99 meV/Ang)
2026-09-07 19:06:54.480 INFO (rank 0): jacobian chunk size automatically set to 16
2026-09-07 19:08:

### Inspect outputs

This will create a folder `run_DATE_TIME_...` (in our case the path is saved in run_path) containing:
* `best_ckpt.pt`  -->  model checkpoint
* `best.json`  -->  train/val/test metrics for the best model
* `config.json`  -->  training configuration
* `log.json`  -->  metrics for all tested models (hyperparameter optimization)

 `best.json` file, which contains info about the **metrics**, **timings**, and **hyperparameters**: see below

In [9]:
!cat {run_path}/best.json

{
    "checkpoint": {
        "hash": "da0f2e7ae388565ceb5285ac83c05492",
        "rf_weight_id": 8
    },
    "timings": {
        "cov_coeffs": 85.60891724284738,
        "solve": 0.0672919424250722
    },
    "metrics": {
        "train": {
            "energy_MAE": 0.141870751732325,
            "forces_MAE": 7.581024348735809
        },
        "validation": {
            "energy_MAE": 0.17251996059897418,
            "forces_MAE": 12.350173087347121
        }
    },
    "hyperparameters": {
        "franken": {
            "path_or_id": "mace_mh/0",
            "family": "mace",
            "interaction_block": 2
        },
        "random_features": {
            "rf_type": "gaussian",
            "num_random_features": 4096,
            "length_scale": 20.0,
            "use_offset": true,
            "rng_seed": 1337
        },
        "input_scaler": {
            "scale_by_Z": true
        },
        "solver": {
            "energy_weight": 1.0,
            "forces_weight": 